# 基于层次聚类改进风险平价策略 - 复现研究

本notebook复现国泰君安研报《基于层次聚类改进风险平价策略》(2024-10-31)

**核心内容:**
1. 层次风险平价(HRP)原理与实现
2. 三种HRP变种策略对比
3. 与传统风险平价策略比较

## 1. 导入必要的库

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['font.sans-serif'] = ['SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

from source import (
    DataLoader,
    HierarchicalRiskParity,
    RiskParity,
    Backtest,
    PerformanceEvaluator
)

print('库导入成功!')

## 2. 数据加载

使用tushare获取10个资产的真实市场数据

In [ ]:
# 初始化数据加载器
loader = DataLoader(start_date='20070101', end_date='20240930')

# 加载所有资产数据
data_loaded = loader.load_all_data()

print(f'\n数据加载状态: {data_loaded}')
print(f'成功获取 {len(loader.price_data)} 个资产的数据')

In [ ]:
# 查看资产列表
print('\n资产列表:')
for i, name in enumerate(loader.price_data.keys()):
    print(f'  {i+1}. {name}')

In [ ]:
# 获取日收益率和月收益率
daily_returns = loader.get_returns(freq='daily')
monthly_returns = loader.get_returns(freq='monthly')

print('\n月收益率数据形状:', monthly_returns.shape)
print('数据时间范围:', monthly_returns.index.min(), '至', monthly_returns.index.max())

## 3. 数据预处理与可视化

In [ ]:
# 计算相关性矩阵
corr_matrix = monthly_returns.corr()

# 绘制相关性热力图
plt.figure(figsize=(12, 10))
plt.imshow(corr_matrix.values, cmap='RdYlBu_r', aspect='auto')
plt.colorbar(label='Correlation')
plt.xticks(range(len(corr_matrix.columns)), corr_matrix.columns, rotation=45, ha='right')
plt.yticks(range(len(corr_matrix.columns)), corr_matrix.columns)
plt.title('资产相关性矩阵', fontsize=14)
plt.tight_layout()
plt.savefig('output/correlation_matrix.png', dpi=150)
plt.show()
print('相关性矩阵已保存至: output/correlation_matrix.png')

## 4. 层次聚类分析

In [ ]:
from scipy.cluster.hierarchy import linkage, dendrogram
from scipy.spatial.distance import squareform

# 计算距离矩阵
distance_matrix = np.sqrt(2 * (1 - corr_matrix.values))
np.fill_diagonal(distance_matrix, 0)

# 层次聚类
condensed_dist = squareform(distance_matrix, checks=False)
Z = linkage(condensed_dist, method='single')

# 绘制聚类树状图
plt.figure(figsize=(14, 8))
dendrogram(Z, labels=list(corr_matrix.columns))
plt.title('资产层次聚类树状图', fontsize=14)
plt.xlabel('资产', fontsize=12)
plt.ylabel('距离', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('output/dendrogram.png', dpi=150)
plt.show()
print('聚类树状图已保存至: output/dendrogram.png')

## 5. 策略回测

In [ ]:
# 初始化回测引擎
backtest = Backtest(
    returns_df=monthly_returns,
    transaction_cost=0.0005,  # 双边万分之五
    rebalance_freq='monthly'
)

# 运行所有策略
results = backtest.run_all_strategies()

In [ ]:
# 获取累计收益率
cumulative_returns = backtest.get_cumulative_returns()

# 绘制净值曲线
plt.figure(figsize=(14, 8))
colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']

for i, (name, cumulative) in enumerate(cumulative_returns.items()):
    plt.plot(cumulative.index, cumulative.values, label=name, linewidth=2, color=colors[i])

plt.title('策略净值曲线对比', fontsize=14)
plt.xlabel('日期', fontsize=12)
plt.ylabel('净值', fontsize=12)
plt.legend(loc='upper left', fontsize=10)
plt.grid(True, alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('output/nav_comparison.png', dpi=150)
plt.show()
print('净值曲线已保存至: output/nav_comparison.png')

## 6. 绩效评估

In [ ]:
# 初始化绩效评估器
evaluator = PerformanceEvaluator(backtest.results)

# 计算所有指标
metrics_df = evaluator.evaluate_all()
print('\n===== 绩效指标汇总 =====\n')
print(metrics_df.to_string(index=False))

In [ ]:
# 绘制指标对比图
plt = evaluator.plot_metrics_comparison(figsize=(16, 12))
plt.savefig('output/metrics_comparison.png', dpi=150)
plt.show()
print('指标对比图已保存至: output/metrics_comparison.png')

In [ ]:
# 绘制回撤对比图
plt = evaluator.plot_drawdown(figsize=(14, 8))
plt.savefig('output/drawdown_comparison.png', dpi=150)
plt.show()
print('回撤对比图已保存至: output/drawdown_comparison.png')

In [ ]:
# 获取年度收益
yearly_returns = evaluator.get_yearly_returns()
print('\n===== 年度收益明细 =====\n')
print((yearly_returns * 100).round(2).to_string())

In [ ]:
# 绘制年度收益热力图
plt = evaluator.plot_yearly_returns_heatmap(figsize=(14, 8))
plt.savefig('output/yearly_returns_heatmap.png', dpi=150)
plt.show()
print('年度收益热力图已保存至: output/yearly_returns_heatmap.png')

## 7. 资产权重分析

In [ ]:
# 获取最新权重
weights_history = backtest.get_Weights_history()

for strategy_name in weights_history.keys():
    latest_weights = pd.DataFrame(weights_history[strategy_name]).iloc[-1]
    
    plt.figure(figsize=(12, 6))
    colors = ['#ff6b6b' if w < 0 else '#4ecdc4' for w in latest_weights.values]
    plt.bar(range(len(latest_weights)), latest_weights.values, color=colors)
    plt.xticks(range(len(latest_weights)), latest_weights.index, rotation=45, ha='right')
    plt.title(f'{strategy_name} - 最新资产权重分布', fontsize=14)
    plt.xlabel('资产', fontsize=12)
    plt.ylabel('权重', fontsize=12)
    plt.grid(True, alpha=0.3, axis='y')
    plt.tight_layout()
    plt.savefig(f'output/weights_{strategy_name}.png', dpi=150)
    plt.show()

## 8. 生成完整报告

In [ ]:
# 生成绩效报告
report = evaluator.generate_report(output_path='output/performance_report.csv')

# 保存回测结果
backtest.save_results(filepath='output/backtest_results.csv')

print('\n所有报告已生成完毕!')

## 9. 结论

根据回测结果，层次风险平价(HRP)策略在以下方面表现优异:
- **最大回撤**: 显著低于传统风险平价策略
- **夏普比率**: 风险调整后收益更优
- **卡玛比率**: 单位最大回撤收益更高

层次聚类方法有效解决了多资产高相关性环境下的配置难题。